# Tesina: PLN para sistematizar el corpus de inscripciones del Parque de Mayo

Este cuaderno corre los tres scripts de la tesina sobre la planilla del corpus:

| Script | Qué hace | Archivos que deja en `resultados/` |
|---|---|---|
| `ordenar_corpus.py` | Revisa la planilla, detecta rasgos, compara tokenizadores, cuenta frecuencias y etiqueta con spaCy | 01 a 08 y `resumen_corpus.xlsx` |
| `probar_freeling.py` | Etiqueta con FreeLing y compara con spaCy | 09 y 10 |
| `probar_deteccion_lengua.py` | Compara tres detectores de lengua con la columna LENGUA | 11 y 12 |

**Antes de empezar.** Colab borra todo lo instalado cada vez que se reinicia el entorno de ejecución. En una sesión nueva hay que correr las celdas en orden, desde la 1. Si una celda falla, mirá primero si corriste las anteriores.

El código se descarga de GitHub, que guarda todas sus versiones; el corpus se toma de Drive.

Cómo se lee una celda: lo que empieza con `#` es un comentario (Python y la terminal lo ignoran) y lo que empieza con `!` es un comando de la terminal de Linux, no de Python.

## 1. Instalar FreeLing

Solo hace falta para `probar_freeling.py`; si hoy no vas a usar FreeLing, saltá a la sección 2. Tarda alrededor de un minuto y medio.

FreeLing no se instala con `pip`: es un programa aparte, que se descarga ya compilado para Ubuntu 22.04. Necesita dos bibliotecas que Colab no trae (ICU 70 y Boost 1.74), así que la celda las descarga e instala también. Al final hace una prueba con «te amo»: si aparece `te te PP2CS00` y `amo amar VMIP1S0`, FreeLing está listo.

In [ ]:
%%bash
# %%bash: toda esta celda se ejecuta en la terminal de Linux, no en Python
cd /content                                        # carpeta de trabajo de Colab
B=http://archive.ubuntu.com/ubuntu/pool/main      # B guarda la dirección del repositorio de Ubuntu, para no repetirla

# Descarga FreeLing 4.2.1 compilado para Ubuntu 22.04 (jammy); -q: sin mensajes
wget -q https://github.com/TALP-UPC/FreeLing/releases/download/4.2/freeling-4.2.1-jammy-amd64.deb
# Descarga ICU 70, la biblioteca de Unicode que usa FreeLing
wget -q $B/i/icu/libicu70_70.1-2_amd64.deb
# Descarga las seis partes de Boost 1.74 que usa FreeLing, una por vuelta del bucle
for L in program-options regex filesystem system thread iostreams; do
  wget -q $B/b/boost1.74/libboost-${L}1.74.0_1.74.0-14ubuntu3_amd64.deb
done

# Instala todos los .deb descargados; > /dev/null esconde los mensajes de instalación
apt-get install -y -qq ./*.deb > /dev/null
# FreeLing necesita la configuración regional en_US.UTF-8: si no está, la instala y la genera
locale -a | grep -qi en_us.utf8 || (apt-get install -y -qq locales > /dev/null && locale-gen en_US.UTF-8)

# Prueba: analiza «te amo» con la configuración del español (es.cfg)
echo "te amo" | analyze -f es.cfg --outlv tagged

## 2. Instalar las bibliotecas de Python

Hay que correrla en cada sesión nueva. Si se saltea, `ordenar_corpus.py` se detiene con el error `[E050] Can't find model 'es_core_news_sm'`.

Al terminar, spaCy muestra el aviso «Restart to reload dependencies». Se puede ignorar: los scripts se corren con `!python`, que abre un proceso nuevo y ya encuentra el modelo.

In [ ]:
# spaCy (etiquetado), NLTK (tokenizadores), scikit-learn (clasificador) y openpyxl (leer y escribir Excel)
!pip install -q spacy nltk scikit-learn openpyxl
# Modelo de spaCy para el español: es = español, core = general, news = entrenado con noticias, sm = versión chica
!python -m spacy download es_core_news_sm
# Detectores de lengua; solo los usa probar_deteccion_lengua.py
!pip install -q lingua-language-detector py3langid

## 3. Traer el código desde GitHub

El código vive en GitHub, que guarda todas sus versiones; Drive guarda el corpus. Esta celda descarga los scripts y entra en la carpeta del repositorio.

`VERSION` decide qué versión se descarga: `"main"` es la última. Para repetir un análisis ya citado, poné la versión que figura en el texto (por ejemplo, `"v1.0.0"`).

In [ ]:
import os
os.chdir("/content")                     # trabajar en /content, la carpeta temporal de Colab

REPO = "https://github.com/lumilud/inscripciones-parque-de-mayo.git"   # dirección del repositorio
VERSION = "main"                         # "main" = la última versión; o una versión publicada, p. ej. "v1.0.0"

!rm -rf inscripciones-parque-de-mayo                 # borra una copia descargada antes, si la hubiera
!git clone -q --branch {VERSION} {REPO}             # descarga el código de esa versión
%cd /content/inscripciones-parque-de-mayo
# Muestra qué versión del código quedó descargada: identificador, fecha y descripción del cambio
!git log -1 --format="Versión del código: %h (%ad) %s" --date=short

## 4. Elegir la planilla

Corré **una sola** de las dos celdas:

- **4A**, para trabajar con el corpus completo que está en tu Drive.
- **4B**, para probar con los datos de ejemplo del repositorio (no necesita Drive).

Las dos definen `ARCHIVO`, la planilla que usan las secciones 5 y 6.

In [ ]:
# 4A. Corpus completo desde Google Drive
from google.colab import drive             # herramienta de Colab para conectar Google Drive
drive.mount("/content/drive")              # Drive aparece como carpeta en /content/drive (la primera vez pide permiso)

# Carpeta de Drive del proyecto. Va entre comillas en los comandos porque tiene espacios.
CARPETA = "/content/drive/MyDrive/Proyecto de tesina/TesinaPLN"
!cp "{CARPETA}/corpus.csv" .               # copia la planilla a la carpeta del repositorio (el punto es «acá»)
ARCHIVO = "corpus.csv"                     # nombre de la planilla que van a leer los scripts

In [ ]:
# 4B. Datos de ejemplo del repositorio: 30 producciones anonimizadas
ARCHIVO = "datos/corpus_ejemplo.xlsx"

## 5. Correr `ordenar_corpus.py`

Es el programa principal. El documento «Cómo leer el programa: ordenar_corpus.py comentado» recorre lo que hace paso por paso. Al terminar imprime un resumen (producciones, tokens, formas distintas, anomalías) y deja los archivos 01 a 08 y `resumen_corpus.xlsx` en la carpeta `resultados`, junto con un archivo que dice con qué versión del código se obtuvieron.

In [ ]:
# Corre el programa sobre la planilla; --salida resultados: los archivos se guardan en la carpeta «resultados»
!python ordenar_corpus.py "{ARCHIVO}" --salida resultados
# Anota en resultados/ qué versión del código produjo estos archivos
!git log -1 --format="%h %ad %s" --date=short > resultados/version_del_codigo.txt

# Para material público se puede agregar --anonimizar (reemplaza usuarios y nombres propios).
# spaCy toma por nombres muchas palabras en mayúsculas, así que el resultado hay que revisarlo a mano.

## 6. Correr los otros dos scripts (opcional)

`probar_freeling.py` necesita la sección 1. Conviene correrlo después de la sección 5: si encuentra el archivo 07 de spaCy, arma también la comparación 10 entre las dos herramientas.

In [ ]:
# Etiquetado con FreeLing (salen 09 y 10)
!python probar_freeling.py "{ARCHIVO}" --salida resultados

# Detección de lengua con Lingua y langid, comparada con la columna LENGUA (salen 11 y 12)
!python probar_deteccion_lengua.py "{ARCHIVO}" --salida resultados

## 7. Guardar los resultados

Lo que queda en Colab se pierde al cerrar el entorno. La celda 7A copia los resultados a tu Drive (necesita la 4A); la 7B los descarga a la computadora como .zip.

In [ ]:
# 7A. Copia la carpeta resultados a Drive, junto al corpus; -r: con todo su contenido
!cp -r resultados "{CARPETA}/"

In [ ]:
# 7B. Descarga la carpeta resultados como .zip
from google.colab import files        # herramienta de Colab para descargar archivos
!zip -qr resultados.zip resultados    # comprime la carpeta; -q: sin mensajes, -r: con todo su contenido
files.download("resultados.zip")      # el navegador descarga el .zip

## 8. Guardar en Drive una copia de una versión publicada

Cada versión publicada en GitHub (v0.1.0, v1.0.0...) se puede guardar como .zip en la carpeta `versiones` de Drive, así queda también junto al resto del proyecto. Necesita Drive conectado (celda 4A).

In [ ]:
VERSION_A_GUARDAR = "v0.1.0"     # la versión publicada que querés guardar
DESTINO = "/content/drive/MyDrive/Proyecto de tesina/TesinaPLN/versiones"
os.makedirs(DESTINO, exist_ok=True)   # crea la carpeta versiones si no existe

# Descarga de GitHub el .zip de esa versión y lo guarda en Drive con un nombre que la identifica
!wget -q -O "{DESTINO}/inscripciones-parque-de-mayo-{VERSION_A_GUARDAR}.zip" "https://github.com/lumilud/inscripciones-parque-de-mayo/archive/refs/tags/{VERSION_A_GUARDAR}.zip"
!ls -l "{DESTINO}"                    # lista lo que hay en la carpeta versiones

## 9. Comprobaciones

Las versiones de cada herramienta van en el apartado metodológico: con otra versión, los resultados pueden cambiar.

In [ ]:
import nltk                  # trae NLTK
print(nltk.__version__)      # muestra su versión (comprobada el 4/11/2026: 3.9.1)
import spacy                 # trae spaCy
print(spacy.__version__)     # muestra su versión (comprobada el 4/11/2026: 3.8.18)

La lista de palabras vacías (*stopwords*) de NLTK se usa para filtrar palabras frecuentes. Esta celda comprueba si la lista del español incluye «y»: si se la usara para filtrar, borraría el nexo de las sumatorias («majo y kevin»).

In [ ]:
import nltk
nltk.download('stopwords')                       # descarga las listas de palabras vacías de NLTK
'y' in nltk.corpus.stopwords.words('spanish')    # ¿«y» está en la lista del español? Responde True o False

## 10. Preparar fotos para la prueba con Transkribus

Toma cinco fotos de la zona 6, las agranda al doble y las guarda como JPG a 300 ppp en la carpeta `prueba_transkribus`. Usa las fotos de tu Drive, así que solo funciona con tu cuenta. Para probar otras producciones, cambiá la lista de IDs.

In [ ]:
!pip install -q pillow           # Pillow: biblioteca para abrir y transformar imágenes
from PIL import Image            # trae la herramienta Image de Pillow
import os
from google.colab import drive
drive.mount("/content/drive", force_remount=True)   # conecta Drive; force_remount: aunque ya esté conectado

ORIGEN = "/content/drive/MyDrive/CORPUS/zona 6/perimetro zona 6"                    # carpeta de las fotos originales
SALIDA = "/content/drive/MyDrive/Proyecto de tesina/TesinaPLN/prueba_transkribus"   # carpeta para las copias
os.makedirs(SALIDA, exist_ok=True)   # crea la carpeta de salida; exist_ok=True: no da error si ya existe

for id_prod in ["271", "273", "284", "285", "290"]:      # las que quieras probar
    im = Image.open(f"{ORIGEN}/{id_prod}.png").convert("RGB")         # abre la foto y la pasa a RGB (el JPG no admite transparencia)
    im = im.resize((im.width * 2, im.height * 2), Image.LANCZOS)        # duplica ancho y alto; LANCZOS suaviza al agrandar
    im.save(f"{SALIDA}/{id_prod}.jpg", quality=95, dpi=(300, 300))      # guarda como JPG, calidad 95 y 300 ppp
print(os.listdir(SALIDA))            # lista lo que quedó en la carpeta de salida